# ICA example 

If you use this ICA example as an intermediate step between two processing pipelines make sure you follow the workflow below:

Workflow:
1. Load one or more pipeline part 1 output FIF files from an output folder.
2. Fit and inspect MNE ICA interactively (components, sources, properties).
3. Exclude selected components and save cleaned FIF output for pipeline part 2.

In [ ]:
from pathlib import Path
import mne
from mne.preprocessing import ICA

# Folder containing part-1 output files (*.fif)
INPUT_DIR = Path('./output_dir')

# Folder where ICA-cleaned files are written
OUTPUT_DIR = Path('./output_ica')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
fif_files = sorted(INPUT_DIR.glob('*.fif'))
if not fif_files:
    raise FileNotFoundError(f'No .fif files found in {INPUT_DIR.resolve()}')

print(f'Found {len(fif_files)} FIF file(s):')
for i, f in enumerate(fif_files):
    print(f'[{i}] {f.name}')

In [ ]:
def fit_ica_for_inspection(
    fif_path: Path,
    n_components=0.99,
    method='fastica',
    random_state=97,
):
    raw = mne.io.read_raw_fif(fif_path, preload=True)

    ica = ICA(n_components=n_components, method=method, random_state=random_state)
    ica.fit(raw)

    explained_var_ratio = ica.get_explained_variance_ratio(raw)
    for channel_type, ratio in explained_var_ratio.items():
        print(f"Fraction of {channel_type} variance explained by all components: {ratio}")

    print('Inspect the plots and decide which components to exclude.')
    ica.plot_components(inst=raw)

    return raw, ica

In [ ]:
file_index = 0  # choose from printed list
n_components = 0.99

raw, ica = fit_ica_for_inspection(
    fif_path=fif_files[file_index],
    n_components=n_components,
    method='fastica',
    random_state=97,
)


In [ ]:
# Set excluded components after visual inspection.
# Example: ica.exclude = [0, 2]
ica.exclude = [1, 2, 5]
print('Current exclusion list:', ica.exclude)

# Optional deeper inspection for selected components
ica.plot_overlay(raw, ica.exclude)


In [ ]:
# Apply ICA exclusion and save cleaned output for part 2
cleaned = ica.apply(raw.copy())
input_path = fif_files[file_index]
output_path = OUTPUT_DIR / f'{input_path.stem}_ica.fif'
cleaned.save(output_path, overwrite=True)
print(f'Saved: {output_path.resolve()}')